# 04 · Objective-2 — HCHO hotspots: z-anomaly → consensus → EHSA

**BAH 2026 PS3 · Objective-2 (high-resolution HCHO hotspot maps & source regions).**

TROPOMI HCHO is the noisiest S5P product, so a single statistic is not trustworthy. The
`aqi_india.hotspot` stack: (1) per-season **robust z-anomaly** standardisation
(median/MAD + spatial detrend) to remove India's gradient and seasonal cycle, then a
**≥2-of-3 consensus** of (2) Getis-Ord **Gi\***, (3) **LISA** High-High clusters, and
(4) a 95th-pct / z>2 **exceedance** vote, plus **Emerging Hot Spot Analysis (EHSA)** over an
Oct–Nov window to separate persistent-industrial from new/intensifying burning hotspots.

In [ ]:
import sys, pathlib
# Make the src/ layout importable when running from the notebooks/ folder
# without an editable install. If aqi_india is already installed this is a no-op.
_repo = pathlib.Path.cwd()
for _ in range(4):
    if (_repo / 'src' / 'aqi_india').is_dir():
        sys.path.insert(0, str(_repo / 'src'))
        break
    _repo = _repo.parent
import aqi_india
print('aqi_india', aqi_india.__version__)

## 1. Synthetic HCHO cube (with the injected fire signal)

We build the cube and inject the downwind HCHO enhancement so there is a **real, locatable**
hotspot over the Punjab/Haryana → IGP corridor. The injection records the cluster centroid in
the attributes, which we can use to sanity-check the detection.

In [ ]:
import numpy as np
from aqi_india.sim import synthetic as sim

SEED = 42
grid = sim.make_grid('2023-10-01', n_days=60, res=0.25, seed=SEED)
fires = sim.make_fires(grid, season='oct_nov', seed=SEED)
grid = sim.inject_fire_hcho(grid, fires)
print('injected fire->HCHO centroid: (%.2f N, %.2f E), lag=%s d'
      % (grid.attrs.get('fire_cluster_centroid_lat', float('nan')),
         grid.attrs.get('fire_cluster_centroid_lon', float('nan')),
         grid.attrs.get('fire_hcho_lag_days')))

## 2. Per-season robust climatology (z-anomaly + percentile surface)

`build_climatology(cube, season='post_monsoon')` filters to the season, computes the per-cell
robust **temporal** z-anomaly (`z`), the detrended **spatial** anomaly the Gi\*/LISA detectors
consume (`z_spatial`), the seasonal mean column, and the per-cell 95th-percentile surface. The
robust median/MAD standardisation is what tames the heavy-tailed HCHO noise.

In [ ]:
from aqi_india.hotspot.climatology import build_climatology
import matplotlib.pyplot as plt

clim = build_climatology(grid, season='post_monsoon', var='hcho_col', q=95.0)
print('climatology vars:', list(clim.data_vars), '| season:', clim.attrs['season'])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
clim['hcho_mean'].plot(ax=axes[0], robust=True, cmap='magma'); axes[0].set_title('seasonal mean HCHO')
clim['z_spatial'].plot(ax=axes[1], robust=True, cmap='RdBu_r'); axes[1].set_title('spatial robust anomaly (Gi*/LISA input)')
clim['hcho_p95'].plot(ax=axes[2], robust=True, cmap='viridis'); axes[2].set_title('seasonal 95th-pct surface')
for ax in axes: ax.set_aspect('equal')
plt.tight_layout(); plt.show()

## 3. The three individual votes

We can call each detector directly on the flattened `z_spatial` field to see the votes before
the consensus combines them: **Gi\*** (FDR p<0.01 hotspot mask) and **LISA** (HH cluster cores
plus HL fresh-fire outliers). Both build a KNN spatial-weights graph over the cell centroids.

In [ ]:
from aqi_india.hotspot.getis_ord import getis_ord_gi
from aqi_india.hotspot.lisa import local_morans_i

lon = clim['lon'].values; lat = clim['lat'].values
lon2d, lat2d = np.meshgrid(lon, lat)
zs = clim['z_spatial'].values
finite = np.isfinite(zs)
coords = np.column_stack([lon2d[finite], lat2d[finite]])
vals = zs[finite]

gi = getis_ord_gi(vals, coords, k=8, permutations=499, alpha=0.01)
lisa = local_morans_i(vals, coords, k=8, permutations=499, alpha=0.05)
print('Gi*  hotspot cells :', int(gi.sig.sum()), ' (FDR threshold p<=%.3g)' % gi.fdr_threshold)
print('LISA HH cluster    :', int(lisa.hh_mask.sum()), ' | HL fresh-fire outliers:', int(lisa.hl_mask.sum()))

## 4. Consensus hotspots (≥2-of-3 votes)

`detect_hotspots` runs the climatology + all three votes and returns a GeoDataFrame of the
**confirmed** cells (Gi\* ∧ LISA ∧ exceedance, requiring ≥2). Each row carries the per-vote
flags so the consensus is auditable.

In [ ]:
from aqi_india.hotspot.consensus import detect_hotspots

hot = detect_hotspots(grid, season='post_monsoon', k=8, permutations=499, min_votes=2)
print('confirmed hotspot cells:', len(hot))
hot.head()

In [ ]:
# Map: spatial anomaly background + confirmed consensus cells + the injected centroid.
fig, ax = plt.subplots(figsize=(9, 7.5))
clim['z_spatial'].plot(ax=ax, cmap='RdBu_r', robust=True, add_colorbar=True,
                       cbar_kwargs={'label': 'HCHO spatial robust anomaly'})
if len(hot):
    ax.scatter(hot['lon'], hot['lat'], s=28, facecolor='none', edgecolor='lime',
               linewidth=1.4, label='confirmed hotspot (>=2/3)')
ax.scatter([grid.attrs.get('fire_cluster_centroid_lon')],
           [grid.attrs.get('fire_cluster_centroid_lat')],
           marker='*', s=240, color='black', label='injected fire centroid')
ax.legend(loc='lower left'); ax.set_aspect('equal')
ax.set_title('Consensus HCHO hotspots over India (post-monsoon)')
plt.show()

### Delineate hotspot polygons

`hotspot_polygons` clusters the confirmed centroids (HDBSCAN if available, else haversine
DBSCAN) and wraps each cluster in a polygon (alpha-shape or convex hull), reporting per-cluster
intensity — the mappable Objective-2 deliverable.

In [ ]:
from aqi_india.hotspot.cluster import hotspot_polygons

if len(hot):
    polys = hotspot_polygons(np.column_stack([hot['lon'], hot['lat']]),
                             values=hot['mean_z'].values, eps_km=60, min_samples=3)
    print('hotspot polygons:', len(polys))
    fig, ax = plt.subplots(figsize=(8.5, 7.5))
    clim['hcho_mean'].plot(ax=ax, cmap='magma', robust=True, alpha=0.85)
    if len(polys):
        polys.boundary.plot(ax=ax, color='cyan', linewidth=2)
    ax.set_title('Delineated HCHO hotspot polygons'); ax.set_aspect('equal')
    import matplotlib.pyplot as plt; plt.show()
    display(polys.drop(columns='geometry') if len(polys) else 'no polygons')
else:
    print('no confirmed cells to delineate')

## 5. Emerging Hot Spot Analysis (EHSA) over the Oct–Nov window

EHSA is the headline **temporal** deliverable: it runs Gi\* per time bin, then a modified
(Hamed-Rao) Mann-Kendall trend test + Sen's slope on each cell's Gi\* z-series, and labels every
cell *new / intensifying / persistent / sporadic / ...*. We bin the post-monsoon window into
short periods and feed the per-bin mean HCHO field as the `(n_bins, n_cells)` cube.

In [ ]:
from aqi_india.hotspot.ehsa import emerging_hotspot_analysis, EHSA_CATEGORIES
import pandas as pd

# Restrict to Oct-Nov and bin into ~6-day periods.
months = pd.to_datetime(grid['time'].values).month
octnov = grid.isel(time=np.flatnonzero(np.isin(months, [10, 11])))
hcho = octnov['hcho_col']
tt = pd.to_datetime(hcho['time'].values)
bin_id = ((tt - tt.min()).days // 6)
bins = sorted(np.unique(bin_id))

binned = np.stack([np.nanmean(hcho.isel(time=np.flatnonzero(bin_id == b)).values, axis=0)
                   for b in bins])           # (n_bins, lat, lon)
n_bins = binned.shape[0]
flat = binned.reshape(n_bins, -1)
cell_finite = np.isfinite(flat).all(axis=0)
cube = flat[:, cell_finite]                    # (n_bins, n_cells)
ehsa_coords = np.column_stack([lon2d.ravel()[cell_finite], lat2d.ravel()[cell_finite]])
print('EHSA cube:', cube.shape, '| bins:', n_bins)

In [ ]:
res = emerging_hotspot_analysis(cube, ehsa_coords, bin_labels=[f'b{b}' for b in bins],
                                k=8, permutations=199, gi_alpha=0.05, trend_alpha=0.10)
cats, counts = np.unique(res.category, return_counts=True)
pd.Series(dict(zip(cats, counts))).reindex(EHSA_CATEGORIES).fillna(0).astype(int).to_frame('n_cells')

In [ ]:
# Map the EHSA categories.
code_of = {c: i for i, c in enumerate(EHSA_CATEGORIES)}
cat_grid = np.full(lon2d.size, np.nan)
cat_grid[cell_finite] = [code_of[c] for c in res.category]
cat_grid = cat_grid.reshape(lon2d.shape)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8.5, 7.5))
im = ax.pcolormesh(lon, lat, cat_grid, cmap='tab10', vmin=0, vmax=len(EHSA_CATEGORIES)-1)
cb = fig.colorbar(im, ax=ax, ticks=range(len(EHSA_CATEGORIES)))
cb.ax.set_yticklabels(EHSA_CATEGORIES)
ax.set_title('Emerging Hot Spot Analysis — HCHO (Oct-Nov)'); ax.set_aspect('equal')
plt.show()

## Summary

The consensus detector confirmed HCHO hotspots co-located with the injected Punjab/Haryana →
IGP fire signal, suppressing single-method artefacts by requiring ≥2-of-3 votes on a robustly
standardised field. EHSA then classified the temporal behaviour (new/intensifying vs
persistent) across the burning window — the source-region + temporal-evolution deliverable for
Objective-2. Notebook 05 attributes these hotspots to upwind fires via transport.